In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
def calculate_area_under_metric_curve(metric_history, normalize=False):

    y = np.array(metric_history)

    x = np.arange(len(y))
    
    area = np.trapz(y, x)
    
    if normalize:
        max_possible_area = (len(y) - 1) * 1.0
        if max_possible_area > 0:
            area = area / max_possible_area
            
    return area

In [3]:
def calculate_total_variation(data, normalize=True):

    data = np.array(data)
    
    diffs = np.diff(data)
    
    abs_diffs = np.abs(diffs)
    
    tv_score = np.sum(abs_diffs)
    
    if normalize and len(diffs) > 0:
        tv_score = tv_score / len(diffs)
        
    return tv_score

In [4]:
def plot_combined_history(model_configs, metrics=['loss', 'auc', 'acc'], mode='val', save_dir='./comparison_plots'):

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    sns.set_theme(style="whitegrid", font_scale=1.2)

    for metric in metrics:
        plt.figure(figsize=(8, 8))
        
        print(f"{'='*20} Plotting Combined {metric.upper()} ({mode}) {'='*20}")
        
        has_valid_data = False

        for model_name, config in model_configs.items():
            file_path = config['path']
            
            if not os.path.exists(file_path):
                print(f"[Error] File not found: {file_path}")
                continue

            try:
                history = np.load(file_path, allow_pickle=True).item()
            except Exception as e:
                print(f"[Error] Failed to load {file_path}: {e}")
                continue

            if mode not in history:
                print(f"[Warning] Mode '{mode}' not found in {model_name}")
                continue

            if metric not in history[mode]:
                print(f"[Warning] Metric '{metric}' not found in {model_name} ({mode})")
                continue

            data = history[mode][metric]

            ALC = calculate_area_under_metric_curve(data, normalize=False)
            TV = calculate_total_variation(data, normalize=False)
            
            epochs = range(1, len(data) + 1)

            if metric in ['loss', 'bce', 'var']:
                best_val = np.min(data)
                best_epoch = np.argmin(data) + 1
                best_str = f"Min: {best_val:.4f} ALC: {ALC:.2f} TV: {TV:.2f}"
            else:
                best_val = np.max(data)
                best_epoch = np.argmax(data) + 1
                best_str = f"Max: {best_val:.4f} ALC: {ALC:.2f} TV: {TV:.2f}"
            has_valid_data = True

            plt.plot(
                epochs, data,
                label=f'{model_name} ({best_str})',
                color=config['color'],
                linestyle=config['style'],
                linewidth=config['linewidth'],
                alpha=0.85
            )
            print(f"Loaded {model_name}: Last {metric}={data[-1]:.4f}, Best={best_val:.4f}")

        if not has_valid_data:
            print(f"No valid data found for metric: {metric}")
            plt.close()
            continue

        mode_str = "Validation" if mode == 'val' else "Training"
        plt.xlabel('Epochs', fontsize=14, fontweight='bold')
        plt.ylabel(f'{metric.upper()}', fontsize=14, fontweight='bold')

        if metric == 'acc':
            plt.ylim([0.45, 0.95])
        if metric == 'auc':
            plt.ylim([0.4, 1.0])

        plt.legend(loc="best", fontsize=14, frameon=True, fancybox=True, framealpha=0.9)

        plt.grid(True, linestyle='--', alpha=0.5) 

        save_filename = f"combined_{mode}_{metric}.png"
        save_full_path = os.path.join(save_dir, save_filename)
        plt.savefig(save_full_path, dpi=300, bbox_inches='tight')
        print(f"\nFigure saved to: {save_full_path}\n")
        
        plt.show()

In [ ]:
dataset_name = 'MIMIC'
subset_name = 'val'
class_name = 'Pleural Effusion'
backbone = 'densenet121'
lr = 0.0002 if backbone == 'resnet50' else 0.0005
model_configs = {
    'Baseline': {
        'path': f'',
        'color': 'gray',
        'style': '--',
        'linewidth': 2
    },
    'WID': {
        'path': f'',
        'color': 'blue',
        'style': '-.',
        'linewidth': 2
    },
    'EDL': {
        'path': f'',
        'color': 'orange',
        'style': '-',
        'linewidth': 2
    },
    'DuET': {
        'path': f'',
        'color': 'red',
        'style': '-',
        'linewidth': 2.5
    }
}

plot_combined_history(
    model_configs, 
    metrics=['auc', 'acc', 'bce'],
    mode=subset_name, 
    save_dir= f''
)